# 2024 OPM FEVS – Veri Setinin Okunması ve Hazırlanması

**Veri:** 2024 OPM Federal Employee Viewpoint Survey – Public Release Data File (PRDF, revize: `rv20261007`)
**Kaynak dokümanlar:** PRDF Read Me, Codebook, Indices & Dimensions Guide, 2024 FEVS Technical Report

Dokümanlardan alınan ve bu notebook'ta uygulanan kurallar:

| Kural | Kaynak | Uygulama |
|---|---|---|
| `X` = *Do Not Know / No Basis to Judge* → yüzde hesaplarına **dahil edilmez** | Teknik Rapor s.36 | `X` → `NaN` (ayrıca sayısı raporlanır) |
| Boş hücre = cevaplanmamış soru → hesaplara dahil edilmez | Teknik Rapor s.36 | `NaN` |
| Demografik boşluklar = gizlilik amaçlı **maskeleme** (≥5 kişi kuralı) | Teknik Rapor Bölüm 7 | `NaN` / "Maskelenmiş" — rastgele eksik **değildir** |
| Tüm yüzdeler **ağırlıklı** (`POSTWT`) hesaplanır | Teknik Rapor Bölüm 6 | Ağırlıklı fonksiyonlar |
| **Percent Positive** = 4–5 (Katılıyorum/Kesinlikle, Memnun/Çok memnun, İyi/Çok iyi) | Teknik Rapor s.35 | `>= 4` |
| İndeks skoru = maddelerin **yuvarlanmamış** % pozitiflerinin ortalaması; EEI = 3 alt indeksin ortalaması | Indices Guide | `index_score()` |
| Q16 çoklu seçim (Q16_1–Q16_6, 0/1), indekslere dahil edilmez | Indices Guide | Ayrı tutulur |
| Ajans / LEVEL1: <500 yanıtlı birimler "All Other" (`XX`, `..ZZ`) altında toplanır | Read Me | Kod → isim eşlemesi |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "2024"
assert DATA_DIR.exists(), f"Veri bulunamadı: {DATA_DIR} – data/README.md dosyasına bakın"
CSV_PATH = DATA_DIR / "2024_OPM_FEVS_PRDF_rv20261007.csv"
CODEBOOK_PATH = DATA_DIR / "2024_OPM_FEVS_PRDF_Codebook.xlsx"
CSV_PATH.exists(), CODEBOOK_PATH.exists()

(True, True)

## 1. Codebook'un okunması

In [2]:
cb = pd.read_excel(CODEBOOK_PATH, sheet_name=None)
print({k: v.shape for k, v in cb.items()})

items = cb["Survey Items"]
items.columns = items.columns.str.strip()
items["VARIABLE"] = items["VARIABLE"].str.strip()

# Soru metinleri ve indeks / boyut bilgisi (soru başına tek satır)
item_meta = (items.groupby("VARIABLE", sort=False)
                  .agg(text=("ITEM TEXT", "first"),
                       index=("INDEX", "first"),
                       dimension=("PERFORMANCE DIMENSION", "first"))
                  .reset_index())

# Cevap değer etiketleri: {soru: {değer: etiket}}
value_labels = {v: dict(zip(g["RESPONSE VALUE"].astype(str), g["RESPONSE CHOICES"].str.strip()))
                for v, g in items.groupby("VARIABLE")}

demo = cb["Demographics"]
demo.columns = demo.columns.str.strip()
demo_labels = {v: dict(zip(g["RESPONSE VALUE"].str.strip(), g["RESPONSE CHOICES"].str.strip()))
               for v, g in demo.groupby("VARIABLE")}
demo_text = demo.groupby("VARIABLE")["ITEM TEXT"].first().str.strip().to_dict()

agency_names = dict(zip(cb["Agency Variable"]["CODE"], cb["Agency Variable"]["NAME"]))
level1_names = dict(zip(cb["LEVEL1 Variable"]["CODE"], cb["LEVEL1 Variable"]["NAME"]))
item_meta.head(10)

{'Survey Items': (430, 7), 'Demographics': (23, 5), 'Agency Variable': (36, 2), 'LEVEL1 Variable': (163, 2), 'Other Variables': (2, 2)}


,VARIABLE,text,index,dimension
0,Q1,I am given a real opportunity to improve my sk...,NaN,Performance Dimension: Employee-Focused - Empl...
1,Q2,I feel encouraged to come up with new and bett...,Employee Engagement Index - Intrinsic Work Exp...,NaN
2,Q3,My work gives me a feeling of personal accompl...,Employee Engagement Index - Intrinsic Work Exp...,NaN
3,Q4,I know what is expected of me on the job.,Employee Engagement Index - Intrinsic Work Exp...,NaN
4,Q5,My workload is reasonable.,NaN,Performance Dimension: Employee-Focused - Work...
5,Q6,My talents are used well in the workplace.,Employee Engagement Index - Intrinsic Work Exp...,NaN
6,Q7,I know how my work relates to the agency's goals.,Employee Engagement Index - Intrinsic Work Exp...,NaN
7,Q8,I can disclose a suspected violation of any la...,NaN,Performance Dimension: Foundations - Merit Pri...
8,Q9,I have enough information to do my job well.,NaN,Performance Dimension: Foundations - Performan...
9,Q10,I receive the training I need to do my job well.,NaN,Performance Dimension: Employee-Focused - Empl...


## 2. CSV'nin okunması
Tüm kolonlar önce metin olarak okunur; böylece `X` (Bilmiyorum) ile boş (cevapsız/maskeli) ayrımı kaybolmaz.

In [3]:
raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, na_values=[""])
raw.columns = raw.columns.str.strip()
print(f"Satır: {len(raw):,}   Kolon: {raw.shape[1]}")
print(f"Bellek: {raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")
raw.head()

Satır: 674,207   Kolon: 96


Bellek: 3.22 GB


,RandomID,POSTWT,agency,level1,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,Q16_1,Q16_2,Q16_3,Q16_4,Q16_5,Q16_6,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,Q28,Q29,Q30,Q31,Q32,Q33,Q34,Q35,Q36,Q37,Q38,Q39,Q40,Q41,Q42,Q43,Q44,Q45,Q46,Q47,Q48,Q49,Q50,Q51,Q52,Q53,Q54,Q55,Q56,Q57,Q58,Q59,Q60,Q61,Q62,Q63,Q64,Q65,Q66,Q67,Q68,Q69,Q70,Q71,Q72,Q86,Q87,Q88,Q89,Q90,Q91,DRNO,DHISP,DDIS,DAGEGRP,DSUPER,DFEDTEN,DSEX,DMIL,DLEAVING
0,112970976817,1.5976951118,XX,XXZZ,3,4,4,4,1,NaN,5,4,4,4,4,4,5,4,4,1,1,1,1,0,0,3,4,4,4,5,4,4,4,5,5,4,4,3,3,3,3,3,2,4,4,4,4,3,4,4,1,3,X,3,5,3,4,3,3,4,4,3,3,1,2,3,3,4,4,3,4,3,3,2,2,4,4,2,4,4,4,4,5,5,5,5,2,B,B,B,B,B,B,A,B,D
1,194868625278,1.8283597661,XX,XXZZ,3,3,4,4,4,4,3,3,3,3,3,3,4,3,3,0,0,0,0,0,1,2,3,4,4,4,4,4,4,3,3,3,3,3,2,NaN,3,3,3,3,4,3,2,4,2,3,3,2,3,3,3,3,3,4,4,4,4,4,4,4,4,2,3,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,3,B,B,B,B,B,B,A,A,A
2,152966380283,1.8673976014,XX,XXZZ,2,1,3,3,3,2,4,1,2,2,3,2,3,3,5,0,1,0,0,0,0,2,4,4,5,5,4,5,4,4,4,4,4,4,4,4,4,4,5,1,2,4,2,1,2,2,2,2,3,1,1,2,4,4,4,4,4,4,4,3,2,1,1,1,1,2,1,X,2,1,1,1,1,1,1,1,1,2,3,1,4,4,3,B,B,B,B,A,A,A,A,A
3,193041162980,2.0773588861,XX,XXZZ,4,5,5,5,2,5,5,4,5,5,4,5,5,5,4,0,0,0,0,1,0,4,4,4,4,5,4,4,X,5,5,4,4,4,4,3,4,4,5,4,4,5,4,4,3,4,4,4,4,3,4,3,4,5,5,5,5,4,4,4,5,4,4,4,4,4,4,5,4,4,3,4,4,5,5,4,5,5,5,5,5,5,2,B,B,B,B,A,A,A,A,A
4,146655962451,1.494947404,XX,XXZZ,4,4,4,4,4,4,4,4,4,4,4,4,4,2,5,1,0,0,0,0,0,4,4,5,5,5,5,5,4,5,5,5,5,4,4,3,4,4,4,4,4,3,4,4,3,2,3,4,4,3,4,3,5,5,5,5,5,5,5,5,5,3,3,4,3,3,4,3,3,3,3,4,3,4,4,4,4,5,5,4,4,4,3,B,B,A,B,B,C,A,A,A


## 3. Değişken gruplarının tanımlanması

In [4]:
ID_COLS     = ["RandomID", "POSTWT", "agency", "level1"]
Q16_COLS    = [f"Q16_{i}" for i in range(1, 7)]
LIKERT_COLS = [c for c in raw.columns if c.startswith("Q") and c not in Q16_COLS and c != "Q91"]
DEMO_COLS   = ["DRNO", "DHISP", "DDIS", "DAGEGRP", "DSUPER", "DFEDTEN", "DSEX", "DMIL", "DLEAVING"]

assert set(ID_COLS + Q16_COLS + LIKERT_COLS + ["Q91"] + DEMO_COLS) == set(raw.columns)
print(f"Likert madde sayısı: {len(LIKERT_COLS)}  ->  {LIKERT_COLS[0]} … {LIKERT_COLS[-1]}")

# Gözlenen değerlerin codebook ile uyumu
for c in LIKERT_COLS:
    extra = set(raw[c].dropna().unique()) - set(value_labels[c])
    assert not extra, (c, extra)
for c in DEMO_COLS:
    extra = set(raw[c].dropna().unique()) - set(demo_labels[c])
    assert not extra, (c, extra)
print("Tüm değerler codebook ile uyumlu.")

Likert madde sayısı: 76  ->  Q1 … Q90


Tüm değerler codebook ile uyumlu.


## 4. Tip dönüşümleri ve temizlik

In [5]:
df = pd.DataFrame(index=raw.index)
df["RandomID"] = raw["RandomID"]
df["POSTWT"]   = raw["POSTWT"].astype(float)
df["agency"]   = raw["agency"].astype("category")
df["level1"]   = raw["level1"].astype("category")
df["agency_name"] = raw["agency"].map(agency_names).astype("category")
df["level1_name"] = raw["level1"].map(level1_names).astype("category")

# Likert maddeler: 1–5 sayısal; X (Do Not Know / No Basis to Judge) -> NaN
dk_counts = (raw[LIKERT_COLS] == "X").sum()
for c in LIKERT_COLS:
    df[c] = pd.to_numeric(raw[c].replace("X", np.nan)).astype("Int8")

# Q16 çoklu seçim (0/1)
for c in Q16_COLS:
    df[c] = pd.to_numeric(raw[c]).astype("Int8")

# Q91 tele-çalışma (nominal)
df["Q91"] = pd.Categorical(raw["Q91"].map(value_labels["Q91"]),
                           categories=list(value_labels["Q91"].values()))

# Demografikler: kod -> etiket; boş = maskelenmiş/cevapsız
for c in DEMO_COLS:
    df[c] = pd.Categorical(raw[c].map(demo_labels[c]), categories=list(demo_labels[c].values()))

print(df.dtypes.value_counts())
print(f"Bellek: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
df.head()

Int8        82
category     2
str          1
float64      1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64
Bellek: 168 MB


,RandomID,POSTWT,agency,level1,agency_name,level1_name,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,Q10,Q11,Q12,Q13,Q14,Q15,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,Q28,Q29,Q30,Q31,Q32,Q33,Q34,Q35,Q36,Q37,Q38,Q39,Q40,Q41,Q42,Q43,Q44,Q45,Q46,Q47,Q48,Q49,Q50,Q51,Q52,Q53,Q54,Q55,Q56,Q57,Q58,Q59,Q60,Q61,Q62,Q63,Q64,Q65,Q66,Q67,Q68,Q69,Q70,Q71,Q72,Q86,Q87,Q88,Q89,Q90,Q16_1,Q16_2,Q16_3,Q16_4,Q16_5,Q16_6,Q91,DRNO,DHISP,DDIS,DAGEGRP,DSUPER,DFEDTEN,DSEX,DMIL,DLEAVING
0,112970976817,1.597695,XX,XXZZ,All Other Agencies,All Other Agencies,3,4,4,4,1,<NA>,5,4,4,4,4,4,5,4,4,3,4,4,4,5,4,4,4,5,5,4,4,3,3,3,3,3,2,4,4,4,4,3,4,4,1,3,<NA>,3,5,3,4,3,3,4,4,3,3,1,2,3,3,4,4,3,4,3,3,2,2,4,4,2,4,4,4,4,5,5,5,5,1,1,1,1,0,0,Teleworks infrequently; ranging from as-needed...,White,No,No,40 or Older,Supervisor/Manager/Executive,Eleven to 20 years,Male,No Prior Military Service,"Yes, to take another job outside the Federal G..."
1,194868625278,1.828360,XX,XXZZ,All Other Agencies,All Other Agencies,3,3,4,4,4,4,3,3,3,3,3,3,4,3,3,2,3,4,4,4,4,4,4,3,3,3,3,3,2,<NA>,3,3,3,3,4,3,2,4,2,3,3,2,3,3,3,3,3,4,4,4,4,4,4,4,4,2,3,2,2,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,4,4,0,0,0,0,0,1,"Does not telework; required to be on site, has...",White,No,No,40 or Older,Supervisor/Manager/Executive,Eleven to 20 years,Male,Military Service,No
2,152966380283,1.867398,XX,XXZZ,All Other Agencies,All Other Agencies,2,1,3,3,3,2,4,1,2,2,3,2,3,3,5,2,4,4,5,5,4,5,4,4,4,4,4,4,4,4,4,4,5,1,2,4,2,1,2,2,2,2,3,1,1,2,4,4,4,4,4,4,4,3,2,1,1,1,1,2,1,<NA>,2,1,1,1,1,1,1,1,1,2,3,1,4,4,0,1,0,0,0,0,"Does not telework; required to be on site, has...",White,No,No,40 or Older,Non-Supervisor/Team Leader,Ten years or fewer,Male,Military Service,No
3,193041162980,2.077359,XX,XXZZ,All Other Agencies,All Other Agencies,4,5,5,5,2,5,5,4,5,5,4,5,5,5,4,4,4,4,4,5,4,4,<NA>,5,5,4,4,4,4,3,4,4,5,4,4,5,4,4,3,4,4,4,4,3,4,3,4,5,5,5,5,4,4,4,5,4,4,4,4,4,4,5,4,4,3,4,4,5,5,4,5,5,5,5,5,5,0,0,0,0,1,0,Teleworks infrequently; ranging from as-needed...,White,No,No,40 or Older,Non-Supervisor/Team Leader,Ten years or fewer,Male,Military Service,No
4,146655962451,1.494947,XX,XXZZ,All Other Agencies,All Other Agencies,4,4,4,4,4,4,4,4,4,4,4,4,4,2,5,4,4,5,5,5,5,5,4,5,5,5,5,4,4,3,4,4,4,4,4,3,4,4,3,2,3,4,4,3,4,3,5,5,5,5,5,5,5,5,5,3,3,4,3,3,4,3,3,3,3,4,3,4,4,4,4,5,5,4,4,4,1,0,0,0,0,0,"Does not telework; required to be on site, has...",White,No,Yes,40 or Older,Supervisor/Manager/Executive,More than 20 years,Male,Military Service,No


## 5. Veri kalitesi kontrolleri

In [6]:
print("Benzersiz RandomID:", df["RandomID"].is_unique)
print(f"Toplam yanıtlayan: {len(df):,}  (Teknik rapor: 674,207 tamamlanmış anket)")
print(f"Ağırlık toplamı (temsil edilen evren): {df['POSTWT'].sum():,.0f}")
df["POSTWT"].describe()

Benzersiz RandomID:

 True
Toplam yanıtlayan: 674,207  (Teknik rapor: 674,207 tamamlanmış anket)
Ağırlık toplamı (temsil edilen evren): 1,687,680


count    674207.000000
mean          2.503208
std           1.737646
min           0.667144
25%           1.484475
50%           1.869337
75%           2.739987
max          14.085054
Name: POSTWT, dtype: float64

In [7]:
# Madde bazında: cevapsız (boş) ve Bilmiyorum/Yargılayamam (X) oranları
quality = pd.DataFrame({
    "bos_%": raw[LIKERT_COLS].isna().mean() * 100,
    "X_DK_NBJ_%": dk_counts / len(raw) * 100,
}).round(1)
quality.sort_values("X_DK_NBJ_%", ascending=False).head(10)

,bos_%,X_DK_NBJ_%
Q24,0.7,12.4
Q17,0.2,10.5
Q39,2.3,5.8
Q47,2.1,5.4
Q58,3.6,5.4
Q63,3.4,5.3
Q45,2.6,4.7
Q61,3.1,4.6
Q20,0.6,4.4
Q33,1.6,4.1


In [8]:
# Demografik maskeleme / eksiklik oranları
(df[DEMO_COLS].isna().mean() * 100).round(1).rename("eksik_veya_maskeli_%").to_frame()

,eksik_veya_maskeli_%
DRNO,20.8
DHISP,13.0
DDIS,10.7
DAGEGRP,12.0
DSUPER,6.8
DFEDTEN,6.8
DSEX,22.0
DMIL,6.8
DLEAVING,6.2


In [9]:
# Ajans dağılımı (ağırlıksız yanıtlayan sayısı)
df["agency_name"].value_counts().head(15)

agency_name
Department of Homeland Security                             94177
United States Department of the Army                        61640
Department of Health and Human Services                     59261
Department of Agriculture                                   50695
Department of the Treasury                                  49794
United States Department of the Navy                        47116
OSD, Joint Staff, Defense Agencies, and Field Activities    44115
Social Security Administration                              40016
Department of Justice                                       38897
Department of the Interior                                  36374
United States Department of the Air Force                   28490
Department of Commerce                                      23882
Department of Transportation                                18488
Department of Energy                                        11037
Environmental Protection Agency                              936

## 6. Ağırlıklı analiz fonksiyonları
Raporla aynı yöntem: Percent Positive = 4–5, X ve boşlar paydadan çıkarılır, `POSTWT` ile ağırlıklandırılır.

In [10]:
def pct_positive(data, col, w="POSTWT"):
    m = data[col].notna()
    return 100 * data.loc[m, w].where(data.loc[m, col] >= 4, 0).sum() / data.loc[m, w].sum()

def pnn(data, col, w="POSTWT"):
    """Ağırlıklı pozitif / nötr / negatif yüzdeleri."""
    m = data[col].notna()
    x, wt = data.loc[m, col], data.loc[m, w]
    tot = wt.sum()
    return pd.Series({"Positive": 100 * wt[x >= 4].sum() / tot,
                      "Neutral":  100 * wt[x == 3].sum() / tot,
                      "Negative": 100 * wt[x <= 2].sum() / tot})

INDICES = {
    "EEI - Leaders Lead":   ["Q57", "Q58", "Q59", "Q61", "Q62"],
    "EEI - Supervisors":    ["Q48", "Q50", "Q51", "Q52", "Q54"],
    "EEI - Intrinsic Work": ["Q2", "Q3", "Q4", "Q6", "Q7"],
    "Global Satisfaction":  ["Q46", "Q70", "Q71", "Q72"],
    "Performance Confidence": ["Q20", "Q21", "Q22", "Q23"],
    "Employee Experience (EXI)": ["Q86", "Q87", "Q88", "Q89", "Q90"],
}

def index_scores(data):
    s = {k: np.mean([pct_positive(data, q) for q in v]) for k, v in INDICES.items()}
    s["Employee Engagement (EEI)"] = np.mean([s["EEI - Leaders Lead"], s["EEI - Supervisors"],
                                              s["EEI - Intrinsic Work"]])
    return pd.Series(s).round(1)

gov_scores = index_scores(df)
gov_scores

EEI - Leaders Lead           62.7
EEI - Supervisors            81.3
EEI - Intrinsic Work         74.9
Global Satisfaction          65.4
Performance Confidence       84.3
Employee Experience (EXI)    74.4
Employee Engagement (EEI)    73.0
dtype: float64

## 7. Hızlı doğrulama — madde bazında hükümet geneli sonuçlar
Bu değerler OPM'in yayınladığı 2024 Governmentwide sonuçlarıyla karşılaştırılabilir (PRDF maskeleme nedeniyle küçük farklar olabilir).

In [11]:
likert_summary = (pd.DataFrame({c: pnn(df, c) for c in LIKERT_COLS}).T.round(1)
                  .join(item_meta.set_index("VARIABLE")["text"]))
likert_summary.sort_values("Positive", ascending=False)

,Positive,Neutral,Negative,text
Q90,91.9,6.2,1.9,It is important to me that my work contribute ...
Q53,88.0,8.7,3.4,My supervisor holds me accountable for achievi...
Q20,87.7,10.3,2.0,Employees in my work unit meet the needs of ou...
Q51,87.1,7.1,5.8,My supervisor treats me with respect.
Q11,86.9,8.5,4.6,I am held accountable for the quality of work ...
...,...,...,...,...
Q67,54.3,23.8,21.8,How satisfied are you with your involvement in...
Q57,51.7,22.0,26.3,"In my organization, senior leaders generate hi..."
Q47,49.5,23.7,26.8,I believe the results of this survey will be u...
Q66,47.8,22.8,29.4,Management involves employees in decisions tha...


## 8. Hazırlanmış verinin kaydedilmesi
Pickle formatı pandas veri tiplerini (kategorik, Int8) olduğu gibi korur ve sonraki notebook'larda çok hızlı yüklenir.

In [12]:
OUT = ROOT / "data" / "fevs2024_clean.pkl"
df.to_pickle(OUT)
print(f"Kaydedildi: {OUT}  ({len(df):,} satır, {df.shape[1]} kolon)")

Kaydedildi: ~\fevs_proje\data\fevs2024_clean.pkl  (674,207 satır, 98 kolon)
